In [ ]:
# ===========================
# FMCG A/B Test Project
# ===========================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind
import os

# ---------------------------
# 1. Generate sample sales data
# ---------------------------

np.random.seed(42)  # for reproducibility

stores = [f"store_{i}" for i in range(1,21)]  # 20 stores
dates = pd.date_range(start='2026-01-01', periods=30)  # 30 days

records = []
for store in stores:
    for date in dates:
        # assign group
        group = "discount" if int(store.split('_')[1]) <= 10 else "control"
        # generate daily sales
        base = 1000 if group == "control" else 1050  # discount slightly higher
        daily_sales = np.random.normal(loc=base, scale=50)  # some variation
        records.append([store, date, group, daily_sales])

sales_df = pd.DataFrame(records, columns=['store', 'date', 'group', 'daily_sales'])

# ---------------------------
# 2. Compute average sales per group
# ---------------------------

grouped_sales = sales_df.groupby('group')['daily_sales'].mean()
print("Average sales per group:")
print(grouped_sales)

# ---------------------------
# 3. Plot bar chart
# ---------------------------

plt.figure(figsize=(6,4))
plt.bar(grouped_sales.index, grouped_sales.values, color=['skyblue', 'orange'])
plt.title('Average Daily Sales: Discount vs Control')
plt.ylabel('Average Sales')
plt.show()

# ---------------------------
# 4. Plot daily trend line
# ---------------------------

daily_trend = sales_df.groupby(['date','group'])['daily_sales'].mean().unstack()
daily_trend.plot(figsize=(12,6))
plt.title('Daily Sales Trend: Discount vs Control')
plt.ylabel('Average Daily Sales')
plt.show()

# ---------------------------
# 5. Run independent t-test
# ---------------------------

discount_sales = sales_df[sales_df['group']=='discount']['daily_sales']
control_sales = sales_df[sales_df['group']=='control']['daily_sales']

t_stat, p_val = ttest_ind(discount_sales, control_sales, equal_var=False)
print(f"T-statistic: {t_stat:.2f}")
print(f"P-value: {p_val:.4f}")

if p_val < 0.05:
    print("Result: Discount promotion significantly increased sales!")
else:
    print("Result: No statistically significant difference detected.")

# ---------------------------
# 6. Save summary CSV
# ---------------------------

os.makedirs("data", exist_ok=True)  # create folder if missing
summary = sales_df.groupby('group')['daily_sales'].agg(['mean','std']).reset_index()
summary.to_csv("data/ab_test_summary.csv", index=False)
print("Summary CSV saved to 'data/ab_test_summary.csv'")
